In [ ]:
import os
import yaml
import shutil
import random
from glob import glob

In [ ]:
# Prepare dataset for YOLO training
def prepare_yolo_dataset(data_dir="data", output_dir="yolo_dataset"):
    # Create directory structure
    for split in ['train', 'val', 'test']:
        os.makedirs(f"{output_dir}/{split}/images", exist_ok=True)
        os.makedirs(f"{output_dir}/{split}/labels", exist_ok=True)
    
    # Collect all image-label pairs from country directories
    all_pairs = []
    country_dirs = [d for d in os.listdir(data_dir) if d.startswith('country_')]
    
    for country_dir in country_dirs:
        images_path = f"{data_dir}/{country_dir}/images"
        labels_path = f"{data_dir}/{country_dir}/labels"
        
        for img_file in glob(f"{images_path}/*.jpg") + glob(f"{images_path}/*.png"):
            base_name = os.path.splitext(os.path.basename(img_file))[0]
            label_file = f"{labels_path}/{base_name}.txt"
            if os.path.exists(label_file):
                all_pairs.append((img_file, label_file))
    
    # Split dataset (70% train, 15% val, 15% test)
    random.seed(42)
    random.shuffle(all_pairs)
    
    n = len(all_pairs)
    train_end = int(0.7 * n)
    val_end = int(0.85 * n)
    
    splits = {
        'train': all_pairs[:train_end],
        'val': all_pairs[train_end:val_end],
        'test': all_pairs[val_end:]
    }
    
    # Copy files to respective directories
    for split_name, pairs in splits.items():
        for img_path, lbl_path in pairs:
            shutil.copy(img_path, f"{output_dir}/{split_name}/images/{os.path.basename(img_path)}")
            shutil.copy(lbl_path, f"{output_dir}/{split_name}/labels/{os.path.basename(lbl_path)}")
    
    # Create YAML configuration
    yaml_data = {
        'path': os.path.abspath(output_dir),
        'train': 'train/images',
        'val': 'val/images', 
        'test': 'test/images',
        'nc': 4,
        'names': ['Pothole', 'Alligator Crack', 'Transverse Crack', 'Longitudinal Crack']
    }
    
    with open(f"{output_dir}/data.yaml", 'w') as f:
        yaml.dump(yaml_data, f, sort_keys=False)
    
    return output_dir

# Prepare dataset if not exists
if not os.path.exists('yolo_dataset/data.yaml'):
    prepare_yolo_dataset()